<br><h1>Fake News Detection Part 2</h1><br>

In [11]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import hstack, vstack, save_npz, load_npz
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
import swifter

import gc
from IPython.display import clear_output

### Task 1 - grouping the labels
To group the labels in two groups, we first identified the unique labels used. This also guarantees that the drop before went as wished.

In [12]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv").to_pandas()
df = df[['type', 'content']].rename(columns={'type':'isfake'}) # rename col for binary

real_labels = ['reliable']
fake_labels = ['bias','fake','unreliable','rumor','conspiracy','clickbait','junksci','satire','political','hate']
drop_labels = ['unknown', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)]
df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)

n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=1000, stratify=labels)

# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=1000, stratify=labels[temp_idx])

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

del df
gc.collect();

(722943, 2)
(90368, 2)
(90368, 2)


In [54]:
# Save words for Bert model
df_train['content'].to_csv('data/X_train_raw')
df_val['content'].to_csv('data/X_val_raw')
df_test['content'].to_csv('data/X_test_raw')

### Task 2 - logistic regression
To train the logistic classifier, we started by converting the contents to 10,000 frequency-based columns based on the top words.

**Bag of Words function:** Given a pandas or polars dataframe and potentially an number of tokens  to be included as features, returns a bag of words representation of the input's column 'content' as a scipy.sparse.csr_matrix.

In [ ]:
vectorizer = CountVectorizer(
    max_features=int(10000), 
    analyzer='word', 
    token_pattern=r'\w{1,}',
    ngram_range=(1,2)
)

# Fit the vocabulary on just 30% of the training data
print('Fitting Vectorizer (30% sample)')
sample_series = df_train['content'].sample(frac=0.3, random_state=1)
vectorizer.fit(sample_series)
del sample_series; gc.collect()

In [ ]:
joblib.dump(vectorizer, 'models/vec_content.pkl')

['training_results/vecotorizer.pkl']

In [14]:
vectorizer = joblib.load('models/vec_content.pkl')

In [15]:
# Helper function to transform in small bites
def chunked_transform(df, cols, vectorizers, chunk_size=50_000, save_as=None, progress=False):
    
    bows = []

    for i, col in enumerate(cols):
        print(f'Processing col "{col}"...')

        chunks = []
        n_rows = len(df)
        
        for start in range(0, n_rows, chunk_size):
            if progress:
                sys.stdout.write(f'\r{start}/{n_rows} rows processed...')
                sys.stdout.flush()
            
            chunk_series = df[col].iloc[start : start + chunk_size]
            
            chunk_matrix = vectorizers[i].transform(chunk_series)
            chunks.append(chunk_matrix)
        
        print()
        col_bow = vstack(chunks, format='csr')
        bows.append(col_bow)


    out = hstack(bows, format='csr')
    if save_as:
        save_npz(save_as, out)

    # Join the heavily compressed chunks into one final matrix
    return out

In [ ]:
# Extract labels before we start deleting things
y_train = df_train['isfake']
y_val = df_val['isfake']
y_test = df_test['isfake']


# Vectorize train, then drop the df
print("Transforming train set...")
X_train = chunked_transform(df_train, ['content'], [vectorizer], save_as='data/X_train.npz', progress=True)

print("Transforming val set...")
X_val = chunked_transform(df_val, ['content'], [vectorizer], save_as='data/X_val.npz', progress=True)

print("Transforming test set...")
X_test = chunked_transform(df_test, ['content'], [vectorizer], save_as='data/X_test.npz', progress=True)

# Let the memory be free!!
del df_train; gc.collect()
del df_val; gc.collect()
del df_test; gc.collect()

try:
    del df; gc.collect()
except NameError:
    pass

Transforming train set...
700000/722943 rows processed...
Transforming val set...
50000/90368 rows processed...
Transforming test set...
50000/90368 rows processed...


In [49]:
y_train.to_csv('data/y_train')
y_val.to_csv('data/y_val')
y_test.to_csv('data/y_test')

In [50]:
# Load data
X_train = load_npz('data/X_train.npz')
X_val = load_npz('data/X_val.npz')
X_test = load_npz('data/X_test.npz')

y_train = pd.read_csv('data/y_train.csv')['isfake']
y_val = pd.read_csv('data/y_val.csv')['isfake']
y_test = pd.read_csv('data/y_test.csv')['isfake']

In [ ]:
# Fit classifier
classifier = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=1000)
classifier.fit(X_train, y_train)

# Predicting
preds = classifier.predict(X_val)
print(classification_report(preds, y_val), end='\n\n' + '='*60 + '\n\n\n')

              precision    recall  f1-score   support

           0       0.92      0.86      0.89     23245
           1       0.95      0.97      0.96     67123

    accuracy                           0.94     90368
   macro avg       0.94      0.92      0.93     90368
weighted avg       0.94      0.94      0.94     90368






In [ ]:
joblib.dump(classifier, 'models/Lars_Lykke_Rasmussen.pkl')

['data/logistic_regression.pkl']

### Task 3 - Inclusion of metadata
We start by reading in the data again. Then we treat it as a kaggle problem, starting with most or all columns and strategically excluding the ones that make the loss go up.

In [16]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv").to_pandas()
df = df.rename(columns={'type':'isfake'}) # rename col for binary

# drop columns we deem mostly useless
df.drop(['url','scraped_at','inserted_at','updated_at'], axis=1, inplace=True)

real_labels = ['reliable']
fake_labels = ['bias','fake','unreliable','rumor','conspiracy','clickbait','junksci','satire','political','hate']
drop_labels = ['unknown', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)]
df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)


In [3]:
# Print cols unique counts, and nan fractions
n = len(df)
for col in df:
    print(
        f'Col: {col:<20}'\
        f'#Unique: {len(df[col].unique()):<10}'\
        f'Dtype: {str(df[col].dtype):<10}'
        f'\tnan: {(df[col] == "nan").sum() / n:.3f}'
    )

Col: domain              #Unique: 557       Dtype: str       	nan: 0.000
Col: isfake              #Unique: 2         Dtype: int64     	nan: 0.000
Col: content             #Unique: 728467    Dtype: str       	nan: 0.000
Col: title               #Unique: 736089    Dtype: str       	nan: 0.009
Col: authors             #Unique: 90983     Dtype: str       	nan: 0.450
Col: meta_keywords       #Unique: 239790    Dtype: str       	nan: 0.645
Col: meta_description    #Unique: 373508    Dtype: str       	nan: 0.544
Col: tags                #Unique: 125504    Dtype: str       	nan: 0.782
Col: source              #Unique: 3         Dtype: str       	nan: 0.762


Now we'll convert some of the most popular domains and authors to categorical features for prediction.

In [18]:
# Categorical values
cat_cols = ['domain','authors','source']

# Top 100 values
for col in cat_cols:
    top_vals = df[col].value_counts().index[:99]
    df.loc[~df[col].isin(top_vals), col] = '<OTHER>'

# Aggregate string columns for approximations
agg_cols = ['title','meta_keywords','meta_description','tags']
df['meta_str_agg'] = df[agg_cols].sum(axis=1)
df.drop(agg_cols, axis=1, inplace=True)

In [5]:
for col in df:
    print(
        f'Col: {col:<20}'\
        f'#Unique: {len(df[col].unique()):<10}'\
        f'Dtype: {str(df[col].dtype):<10}'
        f'\tnan: {(df[col] == "nan").sum() / n:.3f}'
    )

Col: domain              #Unique: 100       Dtype: str       	nan: 0.000
Col: isfake              #Unique: 2         Dtype: int64     	nan: 0.000
Col: content             #Unique: 728467    Dtype: str       	nan: 0.000
Col: authors             #Unique: 100       Dtype: str       	nan: 0.450
Col: source              #Unique: 3         Dtype: str       	nan: 0.762
Col: meta_str_agg        #Unique: 778556    Dtype: str       	nan: 0.000


In [19]:
cat_dummies = pd.get_dummies(df[cat_cols], dtype='int')
df = pd.concat([df.drop(cat_cols, axis=1), cat_dummies], axis=1)
df.shape

(903679, 206)

In [20]:
# train-val-test split - IDENTICAL TO BEFORE
n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=1000, stratify=labels)

# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=1000, stratify=labels[temp_idx])

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(722943, 206)
(90368, 206)
(90368, 206)


In [9]:
# Average lengths
avg_content = np.sum([len(x) for x in df_train['content']]) / len(df_train)
avg_meta = np.sum([len(x) for x in df_train['meta_str_agg']]) / len(df_train)

print(f'Average length for content col: {avg_content:.3f}')
print(f'Average length for meta col: {avg_meta:.3f}')

Average length for content col: 1722.175
Average length for meta col: 159.713


In [ ]:
# Use CountVecotorizer again for comparability

# For meta_str_agg, fiit on the whole training set, but with smaller vocab.
vec_meta = CountVectorizer(
    max_features=int(2000), 
    analyzer='word', 
    token_pattern=r'\w{1,}',
    ngram_range=(1,2)
)

# Fit the vocabulary
vec_meta.fit(df_train['meta_str_agg'])
joblib.dump(vec_meta, 'models/vec_meta.pkl')
gc.collect()

In [41]:
# We can use the same vectorizer as before for the content col
# without leakage as we reused the random state.
vec_content = joblib.load('models/vec_content.pkl')
vec_meta = joblib.load('models/vec_meta.pkl')

In [42]:
# Extract labels
y_train = df_train['isfake']
y_val = df_val['isfake']
y_test = df_test['isfake']

y_train.to_csv('data/y_train_meta.csv')
y_val.to_csv('data/y_val_meta.csv')
y_test.to_csv('data/y_test_meta.csv')

In [43]:
# Vectorize train, then drop the df
print("Transforming train set...")
X_train = chunked_transform(
    df_train, 
    ['content', 'meta_str_agg'], 
    [vec_content, vec_meta], 
    save_as='data/X_train_meta.npz', 
    progress=True
)

print("Transforming val set...")
X_val = chunked_transform(
    df_val, 
    ['content', 'meta_str_agg'], 
    [vec_content, vec_meta], 
    save_as='data/X_val_meta.npz', 
    progress=True
)

print("Transforming test set...")
X_test = chunked_transform(
    df_test, 
    ['content', 'meta_str_agg'], 
    [vec_content, vec_meta], 
    save_as='data/X_test_meta.npz', 
    progress=True
)

Transforming train set...
Processing col "content"...
700000/722943 rows processed...
Processing col "meta_str_agg"...
700000/722943 rows processed...
Transforming val set...
Processing col "content"...
50000/90368 rows processed...
Processing col "meta_str_agg"...
50000/90368 rows processed...
Transforming test set...
Processing col "content"...
50000/90368 rows processed...
Processing col "meta_str_agg"...
50000/90368 rows processed...


In [35]:
X_train = load_npz('data/X_train_meta.npz')
X_val = load_npz('data/X_val_meta.npz')
X_test = load_npz('data/X_test_meta.npz')

In [44]:
# Let the memory be free!!
del df_train
del df_val 
del df_test

try:
    del df
except NameError:
    pass

gc.collect()

1494

In [45]:
# Fit classifier
classifier = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=1000)
classifier.fit(X_train, y_train)

joblib.dump(classifier, 'models/Lars_Mette_Lykke_Rasmussen.pkl')

['models/Lars_Mette_Lykke_Rasmussen.pkl']

In [47]:
classifier = joblib.load('models/Lars_Mette_Lykke_Rasmussen.pkl')

In [48]:
# Predicting
preds = classifier.predict(X_val)
print(classification_report(preds, y_val), end='\n\n' + '='*60 + '\n\n\n')

              precision    recall  f1-score   support

           0       0.94      0.90      0.92     22891
           1       0.97      0.98      0.97     67477

    accuracy                           0.96     90368
   macro avg       0.96      0.94      0.95     90368
weighted avg       0.96      0.96      0.96     90368




